In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    ConfusionMatrixDisplay, classification_report, confusion_matrix, 
    mean_absolute_error, r2_score, root_mean_squared_error, f1_score
)
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# 1. Load Data
df = pd.read_csv('cleaned_pollutant_data.csv', low_memory=False)
df.rename(columns={'date': 'Date', 'aqi': 'AQI'}, inplace=True)
df = df.dropna(subset=['Date', 'AQI']).copy()
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

# 2. Cyclical Temporal Features
df['sin_doy'] = np.sin(2 * np.pi * df['Date'].dt.dayofyear / 365.25)
df['cos_doy'] = np.cos(2 * np.pi * df['Date'].dt.dayofyear / 365.25)
df['sin_dow'] = np.sin(2 * np.pi * df['Date'].dt.dayofweek / 7)
df['cos_dow'] = np.cos(2 * np.pi * df['Date'].dt.dayofweek / 7)

# Target (AQI) placed at index 0 for easy extraction
features = ['AQI', 'pm2.5', 'pm10', 'so2', 'nox', 'o3', 'co', 'sin_doy', 'cos_doy', 'sin_dow', 'cos_dow']

# 3. Train-Test Split
split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx].copy()
test_df = df.iloc[split_idx:].copy()

# 4. Independent Scaling & Imputation
train_medians = train_df[features].median()
train_df[features] = train_df[features].ffill().fillna(train_medians)
test_df[features] = test_df[features].ffill().fillna(train_medians)

scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler() # Isolated scaler for inverse transform

train_scaled_X = scaler_X.fit_transform(train_df[features])
train_scaled_y = scaler_y.fit_transform(train_df[['AQI']])

test_scaled_X = scaler_X.transform(test_df[features])
test_scaled_y = scaler_y.transform(test_df[['AQI']])

# -------------------------------------------------------------
# 5. Native Sequence Generation (3D Tensors)
# -------------------------------------------------------------
def create_sequences(X_data, y_data, seq_length):
    X, y = [], []
    for i in range(len(X_data) - seq_length):
        X.append(X_data[i:(i + seq_length)])
        y.append(y_data[i + seq_length]) # Target t+1
    return np.array(X), np.array(y)

SEQ_LENGTH = 7
X_train, y_train = create_sequences(train_scaled_X, train_scaled_y, SEQ_LENGTH)
X_test, y_test = create_sequences(test_scaled_X, test_scaled_y, SEQ_LENGTH)

# 6. LSTM Architecture Compilation
model = Sequential([
    LSTM(64, input_shape=(X_train.shape[1], X_train.shape[2]), return_sequences=False),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse')
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

print("Training LSTM architecture...")
history = model.fit(
    X_train, y_train,
    epochs=100, batch_size=32,
    validation_split=0.1, callbacks=[early_stop], verbose=1
)

# -------------------------------------------------------------
# 7. Reverse Transformation & R2 Score
# -------------------------------------------------------------
y_pred_scaled = model.predict(X_test)
y_pred = scaler_y.inverse_transform(y_pred_scaled).flatten()
y_test_inv = scaler_y.inverse_transform(y_test).flatten()

print('\n--- LSTM Continuous Performance ---')
print(f'MAE:  {mean_absolute_error(y_test_inv, y_pred):.4f}')
print(f'RMSE: {root_mean_squared_error(y_test_inv, y_pred):.4f}')
print(f'R2:   {r2_score(y_test_inv, y_pred):.4f}') # Explicit R2 Print

# -------------------------------------------------------------
# 8. Regulatory Categorization & F1 Score
# -------------------------------------------------------------
CATEGORY_BINS = [-np.inf, 50, 100, 150, 200, 300, np.inf]
CATEGORY_LABELS = ['Good', 'Fair', 'Unhealthy for Sensitive Groups', 'Very Unhealthy', 'Acutely Unhealthy', 'Emergency']

y_test_cat = pd.cut(y_test_inv, bins=CATEGORY_BINS, labels=CATEGORY_LABELS, right=True).astype(str)
y_pred_cat = pd.cut(y_pred, bins=CATEGORY_BINS, labels=CATEGORY_LABELS, right=True).astype(str)
present_labels = [label for label in CATEGORY_LABELS if (y_test_cat == label).any()]

weighted_f1 = f1_score(y_test_cat, y_pred_cat, labels=present_labels, average='weighted', zero_division=0)
print(f'\nWeighted F1-Score: {weighted_f1:.4f}\n') # Explicit F1 Print
print(classification_report(y_test_cat, y_pred_cat, labels=present_labels, zero_division=0))

# -------------------------------------------------------------
# 9. Confusion Matrix Plot
# -------------------------------------------------------------
cm = confusion_matrix(y_test_cat, y_pred_cat, labels=present_labels)
fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=present_labels)
disp.plot(cmap='Blues', ax=ax, values_format='d')

plt.title('LSTM AQI Forecast Confusion Matrix', fontsize=11, fontweight='bold', pad=12)
plt.ylabel('Actual Category', fontsize=10)
plt.xlabel('Predicted Category', fontsize=10)
plt.xticks(rotation=25, ha='right', fontsize=8.5)
plt.yticks(fontsize=8.5)
plt.tight_layout()
plt.savefig('lstm_AQI_confusion_matrix.png', dpi=300)
plt.show()

NameError: name 'input_shape' is not defined